<a href="https://colab.research.google.com/github/Shaimaa307/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shaimaa307/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Rule

Pages with high Google Search impressions but low clicks receive a higher action score because they may benefit from CTR improvements.

## Reason Codes

- LOW_CTR_HIGH_IMPRESSIONS
- LOW_SESSIONS
- HIGH_POSITION

In [11]:
!pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

warehouse = "hf://datasets/FlyRank/internship-warehouse"

print("Connected successfully.")

Connected successfully.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT *
FROM read_parquet(
'{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 100000
"""

df = con.sql(query).df()

print(df.shape)
df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(100000, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

The baseline score prioritizes pages with high search impressions, low clicks, and poor average search position.

The output includes:
- A baseline score
- A reason code
- An action label

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

# Replace missing values
df["gsc_impressions"] = df["gsc_impressions"].fillna(0)
df["gsc_clicks"] = df["gsc_clicks"].fillna(0)
df["gsc_sum_position"] = df["gsc_sum_position"].fillna(0)

# Calculate CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Baseline score
df["baseline_score"] = (
    df["gsc_impressions"] * 0.5
    + (1 - df["ctr"]) * 100
    + df["gsc_sum_position"] * 0.1
)

# Reason code
df["reason_code"] = np.where(
    (df["gsc_impressions"] > 100) & (df["ctr"] < 0.05),
    "LOW_CTR_HIGH_IMPRESSIONS",
    np.where(
        df["gsc_sum_position"] > 200,
        "HIGH_POSITION",
        "LOW_PRIORITY"
    )
)

# Action label
df["action_label"] = np.where(
    df["reason_code"] == "LOW_CTR_HIGH_IMPRESSIONS",
    "Improve CTR",
    np.where(
        df["reason_code"] == "HIGH_POSITION",
        "Improve Ranking",
        "Monitor"
    )
)

# Rank
queue = df.sort_values("baseline_score", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(queue[
    [
        "report_date",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].head(10))

      report_date           content_hash_id  baseline_score  \
90707  2026-03-03  content_36e53e9c707674fc    27116.974547   
95548  2026-03-03  content_db1cf8cf217e6051    14985.666975   
96070  2026-03-03  content_99c63c59330193de    12403.400000   
96334  2026-03-03  content_dd5472aea4c7aa91    11534.456654   
90559  2026-03-03  content_6aa54d6bbdbf6f24    10668.400000   
91834  2026-03-03  content_559cdd76da9306de    10416.600000   
90325  2026-03-03  content_573804af4f4fa09f    10359.200000   
40079  2026-03-02  content_91487e6f59e8e82f    10309.400000   
96039  2026-03-03  content_877c08cf2d1997a1    10018.144690   
90275  2026-03-03  content_8ed3506b84b0a2ad     9769.806147   

                    reason_code action_label  
90707  LOW_CTR_HIGH_IMPRESSIONS  Improve CTR  
95548  LOW_CTR_HIGH_IMPRESSIONS  Improve CTR  
96070  LOW_CTR_HIGH_IMPRESSIONS  Improve CTR  
96334  LOW_CTR_HIGH_IMPRESSIONS  Improve CTR  
90559  LOW_CTR_HIGH_IMPRESSIONS  Improve CTR  
91834  LOW_CTR_HIGH_IMPR

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

The table below reviews the top 20 ranked pages produced by the baseline rule.

Each row includes:
- Suggested action
- Reason code
- Confidence note
- What would make the recommendation wrong

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "Medium - based on impressions, clicks, and search position."
)

top20["what_would_make_it_wrong"] = (
    "Recent content changes, seasonal traffic, or missing analytics data."
)

review = top20[
    [
        "report_date",
        "content_hash_id",
        "baseline_score",
        "action_label",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

review

,report_date,content_hash_id,baseline_score,action_label,reason_code,confidence_note,what_would_make_it_wrong
90707,2026-03-03,content_36e53e9c707674fc,27116.974547,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
95548,2026-03-03,content_db1cf8cf217e6051,14985.666975,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
96070,2026-03-03,content_99c63c59330193de,12403.400000,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
96334,2026-03-03,content_dd5472aea4c7aa91,11534.456654,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
90559,2026-03-03,content_6aa54d6bbdbf6f24,10668.400000,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
91834,2026-03-03,content_559cdd76da9306de,10416.600000,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
90325,2026-03-03,content_573804af4f4fa09f,10359.200000,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
40079,2026-03-02,content_91487e6f59e8e82f,10309.400000,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
96039,2026-03-03,content_877c08cf2d1997a1,10018.144690,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."
90275,2026-03-03,content_8ed3506b84b0a2ad,9769.806147,Improve CTR,LOW_CTR_HIGH_IMPRESSIONS,"Medium - based on impressions, clicks, and sea...","Recent content changes, seasonal traffic, or m..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks

Some pages may receive a high score because of high search impressions, even if they are already performing well. The baseline rule is simple and cannot capture every situation.

## Leakage check

The rule only uses current search signals:
- gsc_impressions
- gsc_clicks
- gsc_sum_position

It does not use future information, product flags, or label-derived features.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Top reason codes:")
print(queue["reason_code"].value_counts())

print("\nMissing GSC values:")
print(queue["gsc_data_available"].isna().sum())

print("\nLeakage check:")
print("No future windows used.")
print("No product flags used.")
print("No label-derived features used.")


Top reason codes:
reason_code
LOW_PRIORITY                80737
HIGH_POSITION                9857
LOW_CTR_HIGH_IMPRESSIONS     9406
Name: count, dtype: int64

Missing GSC values:
0

Leakage check:
No future windows used.
No product flags used.
No label-derived features used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.